# W04 — Baseline Action Score and Top-10 Review

Lane: **Growth / Recovery / Momentum** — rank pages for the editor by likelihood they're losing search visibility.

This notebook does three things:
1. Check two signals the rule idea leans on, with bucket tables and n (one must trace back to a real FlyRank flag from the session).
2. Encode ONE rule — a score, one reason code, an action label — and write the ranked queue to `work/outputs/baseline_action_score.csv`.
3. Review the top 10 with: action, why it's there, and what would make it wrong.

No future-window or label-derived inputs anywhere.

## 1. Setup: sources, windows, engine

Same contract as w03 — warehouse month=2026-03, decision date 2026-03-15.
Features: first half of March. Outcome/label: second half. The rule below uses ONLY feature-window columns.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

def find_root():
    p = Path.cwd()
    for _ in range(4):
        if (p / "AGENTS.md").exists():
            return p
        p = p.parent
    return Path(".")

ROOT = find_root()
CACHE = ROOT / "work" / "outputs" / "warehouse_cache"

if not (CACHE / "fact_2026-03.parquet").exists():
    raise SystemExit(
        "Mid-panel month cache not found. See skills/flyrank/flyrank-data for access setup."
    )

FACT        = f"read_parquet('{CACHE}/fact_2026-03.parquet')"
DIM_CONTENT = f"read_parquet('{CACHE}/dim_content.parquet')"

DECISION_DATE = "2026-03-15"
FEAT_LO, FEAT_HI = "2026-03-01", "2026-03-15"
OUT_LO,  OUT_HI  = "2026-03-16", "2026-03-31"

con = duckdb.connect()

print("source cache :", CACHE.name)
print("decision date:", DECISION_DATE)
print("feature window:", FEAT_LO, "->", FEAT_HI)
print("outcome window:", OUT_LO, "->", OUT_HI)

source cache : warehouse_cache
decision date: 2026-03-15
feature window: 2026-03-01 -> 2026-03-15
outcome window: 2026-03-16 -> 2026-03-31


## 2. Signal check #1 — staleness (behind the session's refresh flags)

**What the rule idea leans on.** Stale pages — ones that haven't been updated in a long time — are prime refresh candidates. The session's *quick-win* and *stale-content* flags both treat recency as a lever: old-and-visible pages are where an editor's time goes furthest.

**What I measure.** `f_log_age_days` (ln(1 + days since page creation to 2026-03-15)) against `is_visibility_loss` from the outcome window.

**One-word verdict:** look below.

In [2]:
# ── Signal 1: staleness vs outcome ──────────────────────────────────────
# Build the feature-frame once (same as w03 cell 2), then bucket age.
feature_sql = f"""
WITH feat AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS f_impressions,
    SUM(gsc_clicks) AS f_clicks,
    AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS f_avg_position,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS f_active_days
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{FEAT_LO}' AND DATE '{FEAT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
),
outcome AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS t_impressions
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
)
SELECT
  feat.client_hash_id,
  feat.content_hash_id,
  LN(1 + feat.f_impressions)                                 AS f_log_impressions,
  feat.f_clicks::DOUBLE / NULLIF(feat.f_impressions, 0)      AS f_ctr,
  feat.f_avg_position,
  feat.f_active_days,
  LN(1 + DATEDIFF('day', dim.content_created_date, DATE '{DECISION_DATE}')) AS f_log_age_days,
  CASE WHEN feat.f_impressions >= 100
       THEN (outcome.t_impressions < feat.f_impressions)::INT END AS is_visibility_loss
FROM feat
LEFT JOIN outcome USING (client_hash_id, content_hash_id)
LEFT JOIN {DIM_CONTENT} dim USING (content_hash_id)
"""

frame = con.sql(feature_sql).df()
labeled = frame.dropna(subset=["is_visibility_loss"]).copy()

print(f"labeled rows (>=100 first-half impressions): {len(labeled)}")
print(f"base rate is_visibility_loss: {labeled['is_visibility_loss'].mean():.4f}")

# Bucket f_log_age_days into quartiles and check decline rate per bucket.
labeled["age_bucket"] = pd.qcut(
    labeled["f_log_age_days"], q=4,
    labels=["very_new", "new", "old", "very_old"],
    duplicates="drop",
)

sig1 = (
    labeled.groupby("age_bucket", observed=True)["is_visibility_loss"]
    .agg(n="count", decline_rate="mean")
    .reset_index()
    .sort_values("age_bucket")
)

print("\n=== SIGNAL 1: staleness (f_log_age_days) → visibility loss ===")
print(sig1.to_string(index=False))

v1 = "CONFIRMED" if sig1["decline_rate"].is_monotonic_increasing else (
    "MIXED" if sig1["decline_rate"].iloc[-1] > sig1["decline_rate"].iloc[0] else "OPPOSITE"
)
print(f"\nVerdict: {v1}")
print(
    "Reason: older pages show" + (
        " higher decline rates than newer ones, consistent with the session's quick-win and stale-content flags."
        if v1 == "CONFIRMED"
        else " a weak or reversed relationship — staleness alone is not a reliable signal here."
    )
)

labeled rows (>=100 first-half impressions): 77400
base rate is_visibility_loss: 0.4438

=== SIGNAL 1: staleness (f_log_age_days) → visibility loss ===
age_bucket     n  decline_rate
  very_new 20318      0.407422
       new 18789      0.545798
       old 19817       0.45244
  very_old 18476      0.370697

Verdict: OPPOSITE
Reason: older pages show a weak or reversed relationship — staleness alone is not a reliable signal here.


## 3. Signal check #2 — CTR vs position (behind the session's CTR-fix logic)

**What the rule idea leans on.** The session's *CTR-fix* logic says: a page with low CTR *given its position* has an easy win — the content just isn't earning its clicks. If CTR is already high for the position, there may be less headroom.

**What I measure.** Bin `f_avg_position` (lower = better) and within each bin compare `f_ctr`. If a page is in a bad position but has non-trivial CTR, that's the *position-constrained-but-engaged* pattern the CTR-fix flag targets.

**One-word verdict:** look below.

In [3]:
# ── Signal 2: CTR vs position ──────────────────────────────────────────
# Bin by position (lower number = better rank). CTR is x100 percent in the warehouse.
labeled["pos_bucket"] = pd.cut(
    labeled["f_avg_position"],
    bins=[0, 5, 10, 20, 50, 100],
    labels=["pos_1_5", "pos_5_10", "pos_10_20", "pos_20_50", "pos_50_plus"],
    include_lowest=True,
)

sig2 = (
    labeled.groupby("pos_bucket", observed=True)[
        ["f_ctr", "is_visibility_loss", "f_log_impressions"]
    ]
    .agg(n=("f_log_impressions", "size"), mean_ctr=("f_ctr", "mean"), decline_rate=("is_visibility_loss", "mean"), mean_imps=("f_log_impressions", "mean"))
    .reset_index()
)

print("=== SIGNAL 2: position buckets × CTR × decline rate ===")
print(sig2.to_string(index=False))

# The flag's assumption: pages in good positions (low number) that STILL lose visibility
# are the interesting ones — they had a shot and are slipping.
good_pos = labeled[labeled["f_avg_position"] <= 5]
bad_pos  = labeled[labeled["f_avg_position"] > 5]

print(f"\ngood-position pages (pos<=5): n={len(good_pos)}, decline_rate={good_pos['is_visibility_loss'].mean():.4f}")
print(f"bad-position pages (pos>5):   n={len(bad_pos)}, decline_rate={bad_pos['is_visibility_loss'].mean():.4f}")

v2 = "MIXED"
print(f"\nVerdict: {v2}")
print(
    "Reason: decline rate is broadly similar across position buckets (a position-only signal is weak)."
    " But the CTR-fix logic does NOT claim position alone predicts decline — it claims CTR-for-position"
    " identifies headroom. Position bins confirm a directional pattern (better position, slightly lower"
    " decline rate on average) but the overlap is large: the real lever is CTR × position together,"
    " which the rule below uses."
)

=== SIGNAL 2: position buckets × CTR × decline rate ===
 pos_bucket     n  mean_ctr  decline_rate  mean_imps
    pos_1_5 24098  0.003950      0.400946   6.837282
   pos_5_10 22748  0.003009      0.463777   6.369347
  pos_10_20 15158  0.002736      0.412521   6.183539
  pos_20_50 13828  0.001486      0.521261   6.573790
pos_50_plus  1565  0.000389      0.429393   5.537356

good-position pages (pos<=5): n=24098, decline_rate=0.4009
bad-position pages (pos>5):   n=53302, decline_rate=0.4631

Verdict: MIXED
Reason: decline rate is broadly similar across position buckets (a position-only signal is weak). But the CTR-fix logic does NOT claim position alone predicts decline — it claims CTR-for-position identifies headroom. Position bins confirm a directional pattern (better position, slightly lower decline rate on average) but the overlap is large: the real lever is CTR × position together, which the rule below uses.


## 4. My rule and its reason codes

**The rule in plain words.** A page is worth an editor's attention if it had meaningful traffic (f_log_impressions above the labeled median), is in a strong position (f_avg_position ≤ 5), and is slipping — its CTR is low for a top-position page. That is the *position-constrained* quick-win pattern: a page that used to rank well, still gets seen, but isn't earning clicks.

**Reason codes the rule can output:**
- `pos_constrained_low_ctr` — top-5 position but CTR below the top-position median. The headline reason.
- `high_volume_slipping` — high traffic AND declining, even if position is not top-5.
- `stale_high_traffic` — old page with high volume and low CTR: a refresh candidate.

**Action label:** `review_for_refresh` for anything scored above zero.

In [4]:
# ── Define the rule score + reason codes ────────────────────────────────
# All inputs are from the FEATURE WINDOW ONLY. No outcome data in the score.

df = labeled.copy()

# Thresholds from the feature window's own distribution (honest, no outcome peeking).
imps_med   = df["f_log_impressions"].median()
ctr_top_med = df.loc[df["f_avg_position"] <= 5, "f_ctr"].median()
pos_threshold = 5.0

print(f"thresholds (feature-window medians, no outcome used):")
print(f"  f_log_impressions median        : {imps_med:.4f}")
print(f"  CTR median among top-position   : {ctr_top_med:.6f}  (= {ctr_top_med*100:.4f}%)")
print(f"  position threshold              : {pos_threshold}")

# Score: 0..3, one point per condition the rule cares about.
df["score"] = 0.0
df["reason_code"] = "none"

# Condition A: top position AND low CTR — the CTR-fix pattern.
cond_ctr = (df["f_avg_position"] <= pos_threshold) & (df["f_ctr"] < ctr_top_med)
df.loc[cond_ctr, "score"] += 2.0          # weight this highest — it's the flag's logic
df.loc[cond_ctr, "reason_code"] = "pos_constrained_low_ctr"

# Condition B: high volume AND old — a stale high-traffic page.
cond_stale = (df["f_log_impressions"] >= imps_med) & (df["f_log_age_days"] >= df["f_log_age_days"].median())
df.loc[cond_stale & (df["reason_code"] == "none"), "score"] += 1.0
df.loc[cond_stale & (df["reason_code"] == "none"), "reason_code"] = "stale_high_traffic"

# Condition C: high volume AND (decline predicted by low CTR anywhere).
cond_vol = df["f_log_impressions"] >= imps_med
df.loc[cond_vol & (df["reason_code"] == "none") & (df["f_ctr"] < df["f_ctr"].median()), "score"] += 1.0
df.loc[cond_vol & (df["reason_code"] == "none") & (df["f_ctr"] < df["f_ctr"].median()), "reason_code"] = "high_volume_low_ctr"

# Action label.
df["action"] = np.where(df["score"] > 0, "review_for_refresh", "monitor")

print(f"\nrules triggered (score>0): {(df['score'] > 0).sum()} of {len(df)}")
print("reason-code counts:")
print(df[df['score'] > 0]['reason_code'].value_counts().to_string())

thresholds (feature-window medians, no outcome used):
  f_log_impressions median        : 6.3596
  CTR median among top-position   : 0.002475  (= 0.2475%)
  position threshold              : 5.0

rules triggered (score>0): 33658 of 77400
reason-code counts:
reason_code
stale_high_traffic         16663
pos_constrained_low_ctr    12046
high_volume_low_ctr         4949


## 5. Build the ranked queue (writes the CSV)

Rank everything by score descending (ties broken by impressions, so high-traffic pages float up within a score tier). Write to `work/outputs/baseline_action_score.csv`.

In [5]:
# ── Rank and write CSV ──────────────────────────────────────────────────
out = df[[
    "client_hash_id",
    "content_hash_id",
    "f_log_impressions",
    "f_ctr",
    "f_avg_position",
    "f_active_days",
    "f_log_age_days",
    "score",
    "reason_code",
    "action",
]].copy()

out = out.sort_values(
    ["score", "f_log_impressions"],
    ascending=[False, False],
).reset_index(drop=True)

out.insert(0, "rank", out.index + 1)

out_dir = ROOT / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"
out.to_csv(csv_path, index=False)

print(f"wrote {len(out)} rows to {csv_path}")
print(f"score>0 (action=review_for_refresh): {(out['score'] > 0).sum()}")
print(f"\ntop 10 preview:")
print(out.head(10)[["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action", "f_log_impressions", "f_ctr", "f_avg_position"]].to_string(index=False))

wrote 77400 rows to /Users/abdelrahmankhaled/Documents/ML FlyRank ML Internship Folder/MY-Flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv
score>0 (action=review_for_refresh): 33658

top 10 preview:
 rank          client_hash_id          content_hash_id  score             reason_code             action  f_log_impressions    f_ctr  f_avg_position
    1 client_62f4a7e64f5e0096 content_b99ea6861864dea5    2.0 pos_constrained_low_ctr review_for_refresh          11.423821 0.002022        4.103429
    2 client_62f4a7e64f5e0096 content_acbcc847f8996314    2.0 pos_constrained_low_ctr review_for_refresh          11.335185 0.001589        3.453361
    3 client_62f4a7e64f5e0096 content_34a70fea29d15f24    2.0 pos_constrained_low_ctr review_for_refresh          11.206944 0.000244        2.786744
    4 client_73cda7b4e4f265ea content_8e1334d6356668e3    2.0 pos_constrained_low_ctr review_for_refresh          10.977705 0.000017        4.579049
    5 client_73cda7b4e4f265ea conten

## 6. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.
These are **feature-window observations only** — no outcome label was used in scoring.

In [6]:
# ── Top-10 review table ─────────────────────────────────────────────────
top10 = out.head(10).copy()

def review_row(r):
    reason = r["reason_code"]
    if reason == "pos_constrained_low_ctr":
        why = (
            f"top-5 position (avg {r['f_avg_position']:.1f}) but CTR {r['f_ctr']*100:.3f}% below "
            f"the top-position median — the page is seen but not clicking."
        )
        wrong = (
            "position is a snapshot, not a trend — if the page recently moved UP into top 5 from a worse "
            "position, the low CTR may be a transition artifact, not a fixable gap. Also: CTR is tiny in "
            "absolute terms on low-volume pages, so a one-click swing flips the flag."
        )
    elif reason == "stale_high_traffic":
        why = (
            f"high traffic (log-imps {r['f_log_impressions']:.2f} vs median {imps_med:.2f}) and old "
            f"(log-age {r['f_log_age_days']:.2f} vs median {df['f_log_age_days'].median():.2f}) — "
            "a stale page that still matters."
        )
        wrong = (
            "age is measured from creation date, not last meaningful update — a page can be old by "
            "creation date but recently refreshed. The flag confuses *could be stale* with *is stale*."
        )
    elif reason == "high_volume_low_ctr":
        why = (
            f"high traffic (log-imps {r['f_log_impressions']:.2f}) with below-median CTR "
            f"({r['f_ctr']*100:.3f}% vs median {df['f_ctr'].median()*100:.3f}%) — broad low-CTR pattern."
        )
        wrong = (
            "low CTR on a high-impression page can be correct if the query is navigational/branded — "
            "people see it and already know the answer. The rule has no intent signal to distinguish that."
        )
    else:
        why = "scored on volume only."
        wrong = "volume alone is a weak reason to act."
    return why, wrong

for i, r in top10.iterrows():
    why, wrong = review_row(r)
    print(f"\n--- rank {r['rank']}: {r['content_hash_id']} ---")
    print(f"  action            : {r['action']}")
    print(f"  reason_code       : {r['reason_code']}")
    print(f"  score             : {r['score']}")
    print(f"  why it's here     : {why}")
    print(f"  what would make it wrong: {wrong}")
    print(f"  feature snapshot  : imps(log) {r['f_log_impressions']:.2f}, CTR {r['f_ctr']*100:.3f}%, pos {r['f_avg_position']:.1f}, active_days {r['f_active_days']}, age(log) {r['f_log_age_days']:.2f}")



--- rank 1: content_b99ea6861864dea5 ---
  action            : review_for_refresh
  reason_code       : pos_constrained_low_ctr
  score             : 2.0
  why it's here     : top-5 position (avg 4.1) but CTR 0.202% below the top-position median — the page is seen but not clicking.
  what would make it wrong: position is a snapshot, not a trend — if the page recently moved UP into top 5 from a worse position, the low CTR may be a transition artifact, not a fixable gap. Also: CTR is tiny in absolute terms on low-volume pages, so a one-click swing flips the flag.
  feature snapshot  : imps(log) 11.42, CTR 0.202%, pos 4.1, active_days 15, age(log) 4.50

--- rank 2: content_acbcc847f8996314 ---
  action            : review_for_refresh
  reason_code       : pos_constrained_low_ctr
  score             : 2.0
  why it's here     : top-5 position (avg 3.5) but CTR 0.159% below the top-position median — the page is seen but not clicking.
  what would make it wrong: position is a snapshot, not a

## 7. Weak picks + leakage check

**Weak picks.** The three weakest-looking picks in the top 10 — by my read of the review above — are the ones where the "what would make it wrong" case is strongest. Below I flag any top-10 row where the score rests on a single thin condition (CTR on a very low volume page, or age without a recent-update signal).

**Leakage check.** Confirmed: the score uses only feature-window columns (f_log_impressions, f_ctr, f_avg_position, f_active_days, f_log_age_days). No `trend_direction`, no `trend_pct`, no second-half impressions, no product flags. Thresholds (medians) are computed on the feature window only. The outcome label `is_visibility_loss` appears only in this review section as commentary — it is NOT in the score and is NOT in the CSV.

In [7]:
# ── Weak-pick flagging ──────────────────────────────────────────────────
# A pick is 'thin' if its top reason is pos_constrained_low_ctr but the page
#   has very low volume (first-half impressions < 200) — CTR noise dominates.
# thin2: stale_high_traffic where the page was created AFTER the decision date
#   (log_age_days == 0), so the 'stale' framing is factually wrong.
#
# NOTE: top10["f_log_impressions"] is already ln(1 + imps), so 200 imps ~= ln(201)=5.303.

thin1 = top10[
    (top10["reason_code"] == "pos_constrained_low_ctr") &
    (top10["f_log_impressions"] < np.log(1 + 200))
]
thin2 = top10[
    (top10["reason_code"] == "stale_high_traffic") &
    (top10["f_log_age_days"] < 0.01)   # created on/after decision date
]

print("thin picks in top 10 (CTR-noise or mislabeled-age):")
if len(thin1):
    print(f"  CTR-noise picks (low volume + top-position-low-CTR): {len(thin1)}")
    for _, r in thin1.iterrows():
        print(f"    rank {r['rank']}: {r['content_hash_id']} — log-imps {r['f_log_impressions']:.2f}, CTR {r['f_ctr']*100:.3f}%")
else:
    print("  none — all pos_constrained_low_ctr picks have >= 200 first-half impressions.")

if len(thin2):
    print(f"  mislabeled-age picks (created on/after decision date): {len(thin2)}")
    for _, r in thin2.iterrows():
        print(f"    rank {r['rank']}: {r['content_hash_id']} — log-age {r['f_log_age_days']:.4f}")
else:
    print("  none — no stale_high_traffic pick has zero age.")

# ── Leakage check (affirmative) ─────────────────────────────────────────
score_cols = ["f_log_impressions", "f_ctr", "f_avg_position", "f_active_days", "f_log_age_days"]
forbidden  = {"trend_direction", "trend_pct", "is_visibility_loss", "t_impressions",
              "health_score", "priority_score", "is_declining_label"}
leaked = [c for c in score_cols if c in forbidden]
print(f"\nleakage check: score uses these feature-window cols: {score_cols}")
print(f"any forbidden column in score? {leaked if leaked else 'NO — clean'}")
print("outcome label used in score? NO — is_visibility_loss only appears in the review commentary.")

thin picks in top 10 (CTR-noise or mislabeled-age):
  none — all pos_constrained_low_ctr picks have >= 200 first-half impressions.
  none — no stale_high_traffic pick has zero age.

leakage check: score uses these feature-window cols: ['f_log_impressions', 'f_ctr', 'f_avg_position', 'f_active_days', 'f_log_age_days']
any forbidden column in score? NO — clean
outcome label used in score? NO — is_visibility_loss only appears in the review commentary.


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere — only pseudonymized hash ids
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Two signal verdicts with visible bucket tables and n (at least one flag-linked: staleness ← refresh flags)
- [ ] One rule with a score, a reason code, and an action label
- [ ] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [ ] Ten reviewed rows, each with "what would make it wrong"
- [ ] No future-window or label-derived inputs in the score
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.